### **03 - LLM Explainability Engine**
#### **Generating Audit-Grade Fraud Explanations with Llama 3.1**

This notebook implements an AI-powered explainability layer that translates SHAP (SHapley Additive exPlanations) values into human-readable, regulatory-compliant fraud explanations. The system combines:


1. **SHAP Analysis:** Quantifies each feature's contribution to fraud predictions
2. **LLM Generation:** Llama 3.1-8B generates natural language explanations
3. **Regulatory Compliance:** Outputs match "Notice of Adverse Action" standards

**Use Cases:**

- Customer-facing fraud alerts with clear reasoning
- Compliance documentation for audits
- Model transparency for regulators
- Dispute resolution support

**Key Innovation**: The prompt engineering ensures the LLM correctly interprets positive/negative SHAP values as "anomalies" vs "consistent behavior" to avoid logical contradictions.

###**Environment Setup**

**Install Dependencies**

In [ ]:
%pip install mlflow dagshub

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
%pip install shap

In [ ]:


# DagsHub MLflow tracking
DAGSHUB_REPO_OWNER = "your_repo_name"
DAGSHUB_REPO_NAME ="Credit-Scout-Fraud-Detection"
DAGSHUB_TOKEN ="Your_token"

In [3]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

In [20]:
import pandas as pd
import os
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import pandas as pdz
from trl import SFTTrainer
from transformers import TrainingArguments
import dagshub
from sklearn.model_selection import train_test_split
import builtins
import psutil
import mlflow
import os
from huggingface_hub import snapshot_download, HfApi
from unsloth import FastLanguageModel
import pandas as pd
import tempfile
import dagshub.auth
from huggingface_hub import whoami
import pickle
import numpy as np

In [11]:
with open('shap_metadata.pkl','rb')as f:
  shap_data = pickle.load(f)

In [12]:
background = shap_data['background_sample']
feature_names = shap_data['feature_names']

In [13]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [36]:

X_test = np.load('/content/X_test (1).npy')
y_test = np.load('/content/y_test (2).npy')

In [ ]:
import pickle
import numpy as np

# --- 1. RELOAD SCALER (Fixes NotFittedError) ---
print("Reloading scaler...")
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# --- 2. DEFINE BUSINESS MAPPING (Fixes NameError) ---
BUSINESS_MAP = {
    'step': 'Transaction Timing (Hour)',
    'type_enc': 'Transaction Method',
    'amount': 'Transaction Amount',
    'oldbalanceOrg': 'Origin Account Balance (Pre-Txn)',
    'newbalanceOrig': 'Origin Account Balance (Post-Txn)',
    'oldbalanceDest': 'Recipient Account Balance (Pre-Txn)',
    'newbalanceDest': 'Recipient Account Balance (Post-Txn)',
    'errorBalanceOrig': 'Origin Balance Discrepancy',
    'errorBalanceDest': 'Recipient Balance Discrepancy'
}



In [ ]:
# --- 3. THE "BEHAVIORAL" PROMPT (Fixes the Logical Contradiction) ---
PROMPT_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Senior Model Risk Examiner at a regulated bank.
Your task is to draft a "Notice of Adverse Action" explanation.

### LOGIC GUIDELINES (CRITICAL):
1. **INTERPRETATION:**
   - **Increased Risk (Positive SHAP):** Describe this as a "deviation from established patterns" or an "anomaly."
   - **Lowered Risk (Negative SHAP):** Describe this as "consistent with historical behavior" or "within expected limits."
   - *Example:* If 'Amount' lowered risk, say: "While the amount was substantial, it was consistent with the account's history and therefore did not drive the alert."

2. **NARRATIVE COHERENCE:**
   - If one feature increases risk and another lowers it, explain the nuance. (e.g., "Although the account balance was unusually high (anomaly), the transfer amount itself was typical for this user (consistent).")

3. **TONE:** Professional, objective, and regulatory-focused. No certainty ("This is fraud"). Use "The model identified..."

<|eot_id|><|start_header_id|>user<|end_header_id|>
### Transaction Profile (Real Values):
{data_summary}

### Behavioral Risk Analysis (SHAP):
{shap_summary}

### Compliance-Ready Explanation:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

In [ ]:


# --- 4. THE HELPER FUNCTION (With Auto-Disclaimer) ---
def generate_explanation(sample_idx_in_shap, shap_values, original_samples):
    # A. Inverse Transform & Flatten
    raw_scaled = original_samples.flatten()
    real_values = scaler.inverse_transform(raw_scaled.reshape(1, -1)).flatten()

    if isinstance(shap_values, list):
        vals = shap_values[0]
    else:
        vals = shap_values
    vals = vals.flatten()

    # B. Organize & Sort
    feature_data = []
    for i, col_name in enumerate(feature_names):
        biz_name = BUSINESS_MAP.get(col_name, col_name)
        feature_data.append((biz_name, real_values[i], vals[i]))

    feature_data.sort(key=lambda x: abs(x[2]), reverse=True)
    total_shap_mass = sum([abs(v) for _, _, v in feature_data]) + 1e-9

    # C. Build Context Strings
    data_lines = []
    shap_lines = []

    for name, real_val, shap_val in feature_data[:3]: # Top 3 Drivers
        # Formatting
        if "Amount" in name or "Balance" in name:
            val_str = f"${real_val:,.2f}"
        else:
            val_str = f"{real_val:.2f}"

        contrib_pct = (abs(shap_val) / total_shap_mass) * 100

        # LOGIC INJECTION: We tell the LLM exactly how to interpret the math
        if shap_val > 0:
            logic_hint = "ANOMALY (Increased Risk)"
        else:
            logic_hint = "CONSISTENT BEHAVIOR (Mitigated Risk)"

        data_lines.append(f"- {name}: {val_str}")
        shap_lines.append(
            f"- {name}: {logic_hint} | Contribution: {contrib_pct:.1f}% of decision weight"
        )

    # D. Generate Text
    inputs = tokenizer(
        [PROMPT_TEMPLATE.format(
            data_summary="\n".join(data_lines),
            shap_summary="\n".join(shap_lines)
        )],
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=300, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    cleaned_text = generated_text.split("Compliance-Ready Explanation:")[-1].strip()

    # E. APPEND THE MANDATORY DISCLAIMER (Hard-coded safety)
    final_output = f"""{cleaned_text}

    ***
    **Model Confidence Disclaimer:**
    This explanation reflects statistical patterns learned from historical data and does not constitute definitive evidence of fraud. Risk assessments should be verified by a human review team.
    """
    return final_output

# --- TEST IT ---
print("Generating Final Audit-Grade Report")
explanation = generate_explanation(0, shap_values_calculated, target_sample)
print(explanation)


📝 Generating Final Audit-Grade Report...

assistant

**Notice of Adverse Action Explanation**

**Transaction ID:** [Insert Transaction ID]

**Transaction Details:**

- **Transaction Timing (Hour):** The transaction occurred at 362.00 hours, which is not typical for this account's usual activity period. However, the model's analysis of historical behavior indicates that this timing is consistent with the account's overall pattern of transactions. Despite this anomaly, the transaction timing did not significantly contribute to the overall risk assessment.

- **Transaction Method:** The transaction was initiated through a standard payment method, which is consistent with the account's historical behavior. This method is commonly used by the account holder, and its usage did not drive the alert.

- **Transaction Amount:** The transaction amount of $85,113.85 is substantial but is within the expected limits for this account. The model's analysis indicates that this amount is consistent wit